In [1]:
import random
import math
from collections import deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_finance import candlestick_ohlc
import matplotlib.dates as mdates
from binance import Client

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

/opt/homebrew/Caskroom/miniforge/base/envs/coin/lib/python3.9/site-packages/mpl_finance.py:16: DeprecationWarning: 



    Please use `mplfinance` instead (no hyphen, no underscore).

    To install: `pip install --upgrade mplfinance` 

   For more information, see: https://pypi.org/project/mplfinance/


  __warnings.warn('\n\n  ================================================================='+


In [2]:
client = Client()
klines = np.array(client.get_historical_klines("ETHUSDT", Client.KLINE_INTERVAL_1HOUR, "1 Jan, 2018"))
sample = pd.DataFrame(klines.reshape(-1, 12), dtype=float, columns=['datetime',
                                                                   'open',
                                                                   'high',
                                                                   'low',
                                                                   'close',
                                                                   'volume',
                                                                   'close time',
                                                                   'quote asset volume, number of trades',
                                                                   'number of trades',
                                                                   'taker buy base asset volume',
                                                                   'taker buy quote asset volume',
                                                                   'ignore'])
sample['datetime'] = pd.to_datetime(sample['datetime'], unit='ms')
sample.set_index('datetime', inplace=True)
sample = sample[['open', 'high', 'low', 'close', 'volume']]

In [3]:
sample['ropen_14'] = (sample['open'] - sample['low'].rolling(14).min())/(sample['high'].rolling(14).max() - sample['low'].rolling(14).min())
sample['rhigh_14'] = (sample['high'] - sample['low'].rolling(14).min())/(sample['high'].rolling(14).max() - sample['low'].rolling(14).min())
sample['rlow_14'] = (sample['low'] - sample['low'].rolling(14).min())/(sample['high'].rolling(14).max() - sample['low'].rolling(14).min())
sample['rclose_14'] = (sample['close'] - sample['low'].rolling(14).min())/(sample['high'].rolling(14).max() - sample['low'].rolling(14).min())
sample['rvolume_14'] = (sample['volume'] - sample['volume'].rolling(14).min())/(sample['volume'].rolling(14).max() - sample['volume'].rolling(14).min())

sample['ropen_24'] = (sample['open'] - sample['low'].rolling(24).min())/(sample['high'].rolling(24).max() - sample['low'].rolling(24).min())
sample['rhigh_24'] = (sample['high'] - sample['low'].rolling(24).min())/(sample['high'].rolling(24).max() - sample['low'].rolling(24).min())
sample['rlow_24'] = (sample['low'] - sample['low'].rolling(24).min())/(sample['high'].rolling(24).max() - sample['low'].rolling(24).min())
sample['rclose_24'] = (sample['close'] - sample['low'].rolling(24).min())/(sample['high'].rolling(24).max() - sample['low'].rolling(24).min())
sample['rvolume_24'] = (sample['volume'] - sample['volume'].rolling(24).min())/(sample['volume'].rolling(24).max() - sample['volume'].rolling(24).min())

sample['ropen_168'] = (sample['open'] - sample['low'].rolling(168).min())/(sample['high'].rolling(168).max() - sample['low'].rolling(168).min())
sample['rhigh_168'] = (sample['high'] - sample['low'].rolling(168).min())/(sample['high'].rolling(168).max() - sample['low'].rolling(168).min())
sample['rlow_168'] = (sample['low'] - sample['low'].rolling(168).min())/(sample['high'].rolling(168).max() - sample['low'].rolling(168).min())
sample['rclose_168'] = (sample['close'] - sample['low'].rolling(168).min())/(sample['high'].rolling(168).max() - sample['low'].rolling(168).min())
sample['rvolume_168'] = (sample['volume'] - sample['volume'].rolling(168).min())/(sample['volume'].rolling(168).max() - sample['volume'].rolling(168).min())

sample['ropen_366'] = (sample['open'] - sample['low'].rolling(366).min())/(sample['high'].rolling(366).max() - sample['low'].rolling(366).min())
sample['rhigh_366'] = (sample['high'] - sample['low'].rolling(366).min())/(sample['high'].rolling(366).max() - sample['low'].rolling(366).min())
sample['rlow_366'] = (sample['low'] - sample['low'].rolling(366).min())/(sample['high'].rolling(366).max() - sample['low'].rolling(366).min())
sample['rclose_366'] = (sample['close'] - sample['low'].rolling(366).min())/(sample['high'].rolling(366).max() - sample['low'].rolling(366).min())
sample['rvolume_366'] = (sample['volume'] - sample['volume'].rolling(366).min())/(sample['volume'].rolling(366).max() - sample['volume'].rolling(366).min())

sample['reward'] = np.log(sample['close'].shift(-1) / sample['close'])
sample.dropna(inplace=True)

In [4]:
sample.tail()

,open,high,low,close,volume,ropen_14,rhigh_14,rlow_14,rclose_14,rvolume_14,...,rhigh_168,rlow_168,rclose_168,rvolume_168,ropen_366,rhigh_366,rlow_366,rclose_366,rvolume_366,reward
datetime,,,,,,,,,,,,,,,,,,,,,
2021-12-23 04:00:00,3960.60,3972.00,3944.64,3948.73,6330.4508,0.409836,0.502354,0.280312,0.313504,0.008199,...,0.657911,0.591733,0.601625,0.038508,0.375318,0.388821,0.356414,0.361259,0.017197,-0.000611
2021-12-23 05:00:00,3948.73,3965.34,3941.90,3946.32,6934.5648,0.313504,0.448304,0.258075,0.293946,0.036043,...,0.641802,0.585105,0.595796,0.050175,0.361259,0.380933,0.353169,0.358404,0.022408,0.003534
2021-12-23 06:00:00,3946.33,3962.68,3931.84,3960.29,8129.8927,0.330566,0.479745,0.198358,0.457938,0.091137,...,0.635368,0.560772,0.629587,0.073260,0.358416,0.377782,0.341253,0.374951,0.032717,-0.012371
2021-12-23 07:00:00,3960.58,3963.52,3901.00,3911.60,14331.5136,0.501938,0.526706,0.000000,0.089301,0.376974,...,0.637399,0.486177,0.511816,0.193030,0.375295,0.378777,0.304725,0.317280,0.086205,0.004828
2021-12-23 08:00:00,3911.60,3935.64,3893.23,3930.53,16369.9170,0.145252,0.335336,0.000000,0.294932,0.470925,...,0.569963,0.467383,0.557603,0.232397,0.317280,0.345754,0.295522,0.339702,0.103786,-0.001872


In [11]:
class Environment:
    def __init__(self, sample):
        self.sample = sample[
            ['ropen_14', 'rhigh_14', 'rlow_14', 'rclose_14', 'rvolume_14',
            'ropen_24', 'rhigh_24', 'rlow_24', 'rclose_24', 'rvolume_24',
            'ropen_168', 'rhigh_168', 'rlow_168', 'rclose_168', 'rvolume_168',
            'ropen_366', 'rhigh_366', 'rlow_366', 'rclose_366', 'rvolume_366',
            'reward']
        ].copy()
        self.done = False
        self.episode = None
        self.index = 0

    def reset(self, start_idx=-1):
        self.done = False
        self.index = 0
        if start_idx == -1: start_idx = np.random.randint(0, len(sample) - 24*7)
        self.episode = self.sample[start_idx: start_idx+24*7]

    def step(self, action):
        this_idx = self.episode.index[self.index]
        state = np.array([
            self.episode.loc[this_idx, 'ropen_14'], self.episode.loc[this_idx, 'rhigh_14'], self.episode.loc[this_idx, 'rlow_14'], self.episode.loc[this_idx, 'rclose_14'], self.episode.loc[this_idx, 'rvolume_14'],
            self.episode.loc[this_idx, 'ropen_24'], self.episode.loc[this_idx, 'rhigh_24'], self.episode.loc[this_idx, 'rlow_24'], self.episode.loc[this_idx, 'rclose_24'], self.episode.loc[this_idx, 'rvolume_24'],
            self.episode.loc[this_idx, 'ropen_168'], self.episode.loc[this_idx, 'rhigh_168'], self.episode.loc[this_idx, 'rlow_168'], self.episode.loc[this_idx, 'rclose_168'], self.episode.loc[this_idx, 'rvolume_168'],
            self.episode.loc[this_idx, 'ropen_366'], self.episode.loc[this_idx, 'rhigh_366'], self.episode.loc[this_idx, 'rlow_366'], self.episode.loc[this_idx, 'rclose_366'], self.episode.loc[this_idx, 'rvolume_366']
        ])
        reward = self.episode.loc[this_idx, 'reward'] if action == 1 else 0.

        self.index += 1
        if self.index == len(self.episode): self.done = True

        return (state, reward, self.done)

    # Return the total reward with buy and hold policy
    def basic_policy(self):
        try:
            total_rtn = 0.
            for idx in self.episode.index:
                total_rtn += (self.episode.loc[idx, 'reward'])
        except:
            print("You should reset the environment first")
            raise(AttributeError)

        return np.exp(total_rtn)

In [17]:
# Hyperparameters
EPISODES = 4000
EPS_START = 0.99
EPS_END = 0.05
EPS_DECAY = 300
GAMMA = 0.9
LR = 0.001
BATCH_SIZE = 1024

In [18]:
class DQNAgent():
    def __init__(self):
        self.model = nn.Sequential(
            nn.Linear(20, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 2),
            nn.Sigmoid()
        )
        self.optimizer = optim.Adam(self.model.parameters(),LR)
        self.steps_done = 0
        self.memory = deque(maxlen=24*7)
        self.criterion = nn.SmoothL1Loss()

    def memorize(self, state, action, reward, next_state):
        self.memory.append((torch.FloatTensor([state]), torch.LongTensor([[action]]), torch.FloatTensor([reward]), torch.FloatTensor([next_state])))

    def act(self, state):
        state = torch.FloatTensor(state)
        eps_threshold = EPS_END + (EPS_START - EPS_END)*math.exp(-1.*self.steps_done / EPS_DECAY)
        self.steps_done += 1
        if random.random() > eps_threshold: return self.model(state).data.argmax().item()
        else: return random.randrange(2)

    def learn(self):
        if len(self.memory) < BATCH_SIZE: return
        batch = random.sample(self.memory, BATCH_SIZE)
        states, actions, rewards, next_states = zip(*batch)
        states = torch.cat(states)
        actions = torch.cat(actions)
        rewards = torch.cat(rewards)
        next_states = torch.cat(next_states)

        current_q = self.model(states).gather(1, actions)
        max_next_q = self.model(next_states).detach().max(1)[0]
        expected_q = rewards + (GAMMA + max_next_q)

        #print(current_q.shape, max_next_q.shape)
        loss = self.criterion(current_q.squeeze(), expected_q)
        self.optimizer.zero_grad()
        loss.backward()
        for param in self.model.parameters():
            param.grad.data.clamp_(-1, 1)
        self.optimizer.step()

In [ ]:
env = Environment(sample)
agent = DQNAgent()
basic_history = []
score_history = []
for epi in range(1, EPISODES+1):
    env.reset()
    action = 0
    steps = 0
    total_rewards = 0.
    buy_and_hold = env.basic_policy()
    print(f"[Episode {epi}] buy and hold: {buy_and_hold}")
    basic_history.append(buy_and_hold)
    while True:
        state, reward, done = env.step(action)
        action = agent.act(state)
        next_state, reward, done = env.step(action)
        total_rewards += reward
        if total_rewards < 0.7:
            done = True

        agent.memorize(state, action, reward, next_state)
        agent.learn()

        state = next_state
        steps += 1

        if done:
            print(f"[Episode {epi}] score: {np.exp(total_rewards)}\n")
            score_history.append(np.exp(total_rewards))
            break

[Episode 1] buy and hold: 0.728458575806134
[Episode 1] score: 0.9926121847023063

[Episode 2] buy and hold: 1.0445396145610277
[Episode 2] score: 1.0

[Episode 3] buy and hold: 1.1504623844038984
[Episode 3] score: 1.0

[Episode 4] buy and hold: 0.8254154591144215
[Episode 4] score: 1.0003318461708641

[Episode 5] buy and hold: 0.5734928989969214
[Episode 5] score: 1.0

[Episode 6] buy and hold: 0.9629043014338106
[Episode 6] score: 1.0

[Episode 7] buy and hold: 0.973807829181495
[Episode 7] score: 1.0019995715203884

[Episode 8] buy and hold: 1.1082868867520048
[Episode 8] score: 1.0

[Episode 9] buy and hold: 1.160731672279771
[Episode 9] score: 1.0

[Episode 10] buy and hold: 0.9838553438811736
[Episode 10] score: 1.0

[Episode 11] buy and hold: 0.9993044929753794
[Episode 11] score: 1.0

[Episode 12] buy and hold: 0.6787452166107618
[Episode 12] score: 1.0

[Episode 13] buy and hold: 0.9467114738358133
[Episode 13] score: 0.9416666666666667

[Episode 14] buy and hold: 1.045461272

In [ ]:
plt.figure(figsize=(12, 9))
plt.plot(score_history, color='g', label="DQN agent")
plt.plot(basic_history, color='r', label="buy and hold")
plt.ylabel('total rtn')
plt.legend(loc='best')
plt.show()